**生成式AI使用声明**：对于本次作业，使用生成式AI需遵循与合作相同的政策。与其他协作者一样，每位学生必须独立撰写解决方案，不受交互输出的影响，并在提交中注明协作的性质。使用生成式AI工具实质性完成作业部分内容不符合作业的精神，将违反[荣誉准则](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 挂载Google Drive到Colab虚拟机。
from google.colab import drive
drive.mount('/content/drive')

# TODO: 输入你Drive中保存解压后作业文件夹的文件夹名，
# 例如 'cs231n/assignments/assignment2/'
FOLDERNAME = "cs231n/assignments/assignment2/"
assert FOLDERNAME is not None, "[!] 请输入文件夹名。"

# 挂载Drive后，确保Colab虚拟机的Python解释器
# 可以从其中加载python文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 如果COCO数据集不存在，则下载到你的Drive中。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_coco_dataset.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# 使用RNN进行图像描述
在本练习中，你将实现基础的循环神经网络（RNN），并使用它们训练一个能够为图像生成新颖描述的模型。

In [ ]:
# 设置单元格。
import time, os, json
import numpy as np
import torch
import matplotlib.pyplot as plt

from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.rnn_layers_pytorch import *
from cs231n.captioning_solver_pytorch import CaptioningSolverPytorch
from cs231n.classifiers.rnn_pytorch import CaptioningRNN
from cs231n.coco_utils import load_coco_data, sample_coco_minibatch, decode_captions
from cs231n.image_utils import image_from_url

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # 设置默认绘图大小。
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

import sys
import types
import importlib

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

def rel_error(x, y):
    """ 返回相对误差 """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

# COCO数据集
对于本次练习，我们将使用2014年发布的[COCO数据集](https://cocodataset.org/)，这是图像描述的标准测试平台。该数据集包含80,000张训练图像和40,000张验证图像，每张图像都附有5个由Amazon Mechanical Turk工作人员撰写的描述。

**图像特征。** 我们已经为你预处理了数据并提取了特征。对于所有图像，我们从在ImageNet上预训练的VGG-16网络的fc7层提取了特征，这些特征存储在文件`train2014_vgg16_fc7.h5`和`val2014_vgg16_fc7.h5`中。为了减少处理时间和内存需求，我们使用主成分分析（PCA）将特征的维度从4096降到了512，这些降维后的特征存储在文件`train2014_vgg16_fc7_pca.h5`和`val2014_vgg16_fc7_pca.h5`中。原始图像占用了近20GB的空间，因此我们没有将其包含在下载中。由于所有图像都来自Flickr，我们将训练和验证图像的URL存储在文件`train2014_urls.txt`和`val2014_urls.txt`中。这允许你在可视化时按需下载图像。

**描述。** 处理字符串效率低下，因此我们将使用编码版本的描述。每个单词被分配一个整数ID，使我们能够用整数序列来表示描述。整数ID与单词之间的映射关系在文件`coco2014_vocab.json`中，你可以使用`cs231n/coco_utils.py`文件中的`decode_captions`函数将整数ID的NumPy数组转换回字符串。

**特殊标记。** 我们向词汇表中添加了一些特殊标记，并且已经为你处理了所有与特殊标记相关的实现细节。我们在每个描述的开头和结尾分别添加特殊的`<START>`标记和`<END>`标记。罕见词被替换为特殊的`<UNK>`标记（表示"未知"）。此外，由于我们希望使用包含不同长度描述的小批量进行训练，我们在`<END>`标记之后用特殊的`<NULL>`标记填充较短的描述，并且不计算`<NULL>`标记的损失或梯度。

你可以使用`cs231n/coco_utils.py`文件中的`load_coco_data`函数加载所有COCO数据（描述、特征、URL和词汇表）。运行以下单元格来执行此操作：

In [ ]:
# 从磁盘加载COCO数据到字典中。
# 我们将在本作业的剩余部分使用降维后的特征，
# 但你也可以通过更改下面的标志来自己尝试原始特征。
data = load_coco_data(pca_features=True)

# 打印数据字典中的所有键和值。
for k, v in data.items():
    if type(v) == np.ndarray:
        print(k, type(v), v.shape, v.dtype)
    else:
        print(k, type(v), len(v))

## 查看数据
在处理数据集之前查看示例总是一个好主意。

你可以使用`cs231n/coco_utils.py`文件中的`sample_coco_minibatch`函数从`load_coco_data`返回的数据结构中采样小批量数据。运行以下代码来采样一小批训练数据并显示图像及其描述。多次运行并查看结果有助于你了解数据集。

In [ ]:
# 采样一个小批量并显示图像和描述。
# 如果出现错误，说明URL已不存在，不用担心！
# 你可以重新采样任意多次。
batch_size = 3

captions, features, urls = sample_coco_minibatch(data, batch_size=batch_size)
for i, (caption, url) in enumerate(zip(captions, urls)):
    plt.imshow(image_from_url(url))
    plt.axis('off')
    caption_str = decode_captions(caption, data['idx_to_word'])
    plt.title(caption_str)
    plt.show()

# 循环神经网络
如课堂所述，我们将使用循环神经网络（RNN）语言模型进行图像描述。文件`cs231n/rnn_layers_pytorch.py`包含了循环神经网络所需的不同层类型的实现，文件`cs231n/classifiers/rnn_pytorch.py`使用这些层来实现图像描述模型。

我们将首先在`cs231n/rnn_layers_pytorch.py`中实现不同类型的RNN层。

_可选阅读以深入了解RNN：https://www.deeplearningbook.org/contents/rnn.html#pf7_

# Vanilla RNN：单步前向传播
打开文件`cs231n/rnn_layers_pytorch.py`。该文件实现了循环神经网络中常用的不同类型层的前向传播。注意，由于我们使用pytorch，反向传播将由pytorch的autograd处理。

首先实现函数`rnn_step_forward`，它实现了基础循环神经网络单个时间步的前向传播。完成后运行以下代码检查你的实现。你应该看到数量级为e-8或更小的误差。

In [ ]:
N, D, H = 3, 10, 4

x = torch.from_numpy(np.linspace(-0.4, 0.7, num=N*D).reshape(N, D))
prev_h = torch.from_numpy(np.linspace(-0.2, 0.5, num=N*H).reshape(N, H))
Wx = torch.from_numpy(np.linspace(-0.1, 0.9, num=D*H).reshape(D, H))
Wh = torch.from_numpy(np.linspace(-0.3, 0.7, num=H*H).reshape(H, H))
b = torch.from_numpy(np.linspace(-0.2, 0.4, num=H))

next_h = rnn_step_forward(x, prev_h, Wx, Wh, b).numpy()
expected_next_h = np.asarray([
  [-0.58172089, -0.50182032, -0.41232771, -0.31410098],
  [ 0.66854692,  0.79562378,  0.87755553,  0.92795967],
  [ 0.97934501,  0.99144213,  0.99646691,  0.99854353]])

print('next_h error: ', rel_error(expected_next_h, next_h))

# Vanilla RNN：单步反向传播
由于我们使用pytorch实现了`rnn_step_forward`，我们不需要实现`rnn_step_backward`。我们可以使用数值梯度检查器验证pytorch autograd的反向传播。

然而，如果你想挑战一下，可以尝试自己实现`rnn_step_backward`。不过本次作业不要求这样做。

In [ ]:
from cs231n.rnn_layers_pytorch import rnn_step_forward

# 创建测试输入
np.random.seed(231)
N, D, H = 4, 5, 6
x = torch.from_numpy(np.random.randn(N, D))
h = torch.from_numpy(np.random.randn(N, H))
Wx = torch.from_numpy(np.random.randn(D, H))
Wh = torch.from_numpy(np.random.randn(H, H))
b = torch.from_numpy(np.random.randn(H))

# 启用梯度跟踪并进行rnn前向传播
for tensor in [x, h, Wx, Wh, b]:
  tensor.requires_grad_()
next_h = rnn_step_forward(x, h, Wx, Wh, b)

# 模拟随机上游梯度并使用pytorch的autograd进行反向传播。
dnext_h = torch.from_numpy(np.random.randn(*next_h.shape))
next_h.backward(dnext_h)

# 将梯度收集到单独的numpy数组中
dx = x.grad.detach().numpy()
dh = h.grad.detach().numpy()
dWx = Wx.grad.detach().numpy()
dWh = Wh.grad.detach().numpy()
db = b.grad.detach().numpy()
dnext_h = dnext_h.detach().numpy()

# 将测试输入也转换为numpy数组
x =  x.detach().numpy()
h =  h.detach().numpy()
Wx = Wx.detach().numpy()
Wh = Wh.detach().numpy()
b =  b.detach().numpy()

# 包装我们的前向传播以支持numpy数组输入和输出。我们使用
# `torch.no_grad()`来显式禁用梯度跟踪。
def rnn_step_forward_numpy(x, h, Wx, Wh, b):
  with torch.no_grad():
    return rnn_step_forward(
        torch.from_numpy(x),
        torch.from_numpy(h),
        torch.from_numpy(Wx),
        torch.from_numpy(Wh),
        torch.from_numpy(b),
    ).numpy()

# 计算数值梯度并进行比较。
fx = lambda x: rnn_step_forward_numpy(x, h, Wx, Wh, b)
fh = lambda h: rnn_step_forward_numpy(x, h, Wx, Wh, b)
fWx = lambda Wx: rnn_step_forward_numpy(x, h, Wx, Wh, b)
fWh = lambda Wh: rnn_step_forward_numpy(x, h, Wx, Wh, b)
fb = lambda b: rnn_step_forward_numpy(x, h, Wx, Wh, b)

dx_num = eval_numerical_gradient_array(fx, x, dnext_h)
dh_num = eval_numerical_gradient_array(fh, h, dnext_h)
dWx_num = eval_numerical_gradient_array(fWx, Wx, dnext_h)
dWh_num = eval_numerical_gradient_array(fWh, Wh, dnext_h)
db_num = eval_numerical_gradient_array(fb, b, dnext_h)

# 你应该看到数量级为1e-9或更小的误差
print('dx error: ', rel_error(dx_num, dx))
print('dh error: ', rel_error(dh_num, dh))
print('dWx error: ', rel_error(dWx_num, dWx))
print('dWh error: ', rel_error(dWh_num, dWh))
print('db error: ', rel_error(db_num, db))

# Vanilla RNN：完整前向传播
现在你已经实现了基础RNN单个时间步的前向传播，你将使用它来实现处理整个数据序列的RNN。

在文件`cs231n/rnn_layers_pytorch.py`中，实现函数`rnn_forward`。这应该使用你上面定义的`rnn_step_forward`函数来实现。完成后运行以下代码检查你的实现。你应该看到数量级为`e-7`或更小的误差。

In [ ]:
from cs231n.rnn_layers_pytorch import rnn_forward

N, T, D, H = 2, 3, 4, 5

x = torch.from_numpy(np.linspace(-0.1, 0.3, num=N*T*D).reshape(N, T, D))
h0 = torch.from_numpy(np.linspace(-0.3, 0.1, num=N*H).reshape(N, H))
Wx = torch.from_numpy(np.linspace(-0.2, 0.4, num=D*H).reshape(D, H))
Wh = torch.from_numpy(np.linspace(-0.4, 0.1, num=H*H).reshape(H, H))
b = torch.from_numpy(np.linspace(-0.7, 0.1, num=H))

h = rnn_forward(x, h0, Wx, Wh, b).numpy()
expected_h = np.asarray([
  [
    [-0.42070749, -0.27279261, -0.11074945,  0.05740409,  0.22236251],
    [-0.39525808, -0.22554661, -0.0409454,   0.14649412,  0.32397316],
    [-0.42305111, -0.24223728, -0.04287027,  0.15997045,  0.35014525],
  ],
  [
    [-0.55857474, -0.39065825, -0.19198182,  0.02378408,  0.23735671],
    [-0.27150199, -0.07088804,  0.13562939,  0.33099728,  0.50158768],
    [-0.51014825, -0.30524429, -0.06755202,  0.17806392,  0.40333043]]])
print('h error: ', rel_error(expected_h, h))

# Vanilla RNN：反向传播
与之前一样，我们可以使用数值梯度检查器验证pytorch autograd的反向传播。如果你想的话，也可以尝试自己实现`rnn_step_backward`。不过本次作业不要求这样做。


In [ ]:
from cs231n.rnn_layers_pytorch import rnn_forward

# 创建测试输入
np.random.seed(231)
N, D, T, H = 2, 3, 10, 5
x = torch.from_numpy(np.random.randn(N, T, D))
h0 = torch.from_numpy(np.random.randn(N, H))
Wx = torch.from_numpy(np.random.randn(D, H))
Wh = torch.from_numpy(np.random.randn(H, H))
b = torch.from_numpy(np.random.randn(H))

# 启用梯度跟踪并进行前向传播
for tensor in [x, h0, Wx, Wh, b]:
  tensor.requires_grad_()
h = rnn_forward(x, h0, Wx, Wh, b)

# 模拟随机上游梯度并使用pytorch的autograd进行反向传播。
dh = torch.from_numpy(np.random.randn(*h.shape))
h.backward(dh)

# 将梯度收集到单独的numpy数组中
dx = x.grad.detach().numpy()
dh0 = h0.grad.detach().numpy()
dWx = Wx.grad.detach().numpy()
dWh = Wh.grad.detach().numpy()
db = b.grad.detach().numpy()
dh = dh.detach().numpy()

# 将测试输入也转换为numpy数组
x =  x.detach().numpy()
h0 =  h0.detach().numpy()
Wx = Wx.detach().numpy()
Wh = Wh.detach().numpy()
b =  b.detach().numpy()

# 包装我们的前向传播以支持numpy数组输入和输出。我们使用
# `torch.no_grad()`来显式禁用梯度跟踪。
def rnn_forward_numpy(x, h0, Wx, Wh, b):
  with torch.no_grad():
    return rnn_forward(
        torch.from_numpy(x),
        torch.from_numpy(h0),
        torch.from_numpy(Wx),
        torch.from_numpy(Wh),
        torch.from_numpy(b),
    ).numpy()


fx = lambda x: rnn_forward_numpy(x, h0, Wx, Wh, b)
fh0 = lambda h0: rnn_forward_numpy(x, h0, Wx, Wh, b)
fWx = lambda Wx: rnn_forward_numpy(x, h0, Wx, Wh, b)
fWh = lambda Wh: rnn_forward_numpy(x, h0, Wx, Wh, b)
fb = lambda b: rnn_forward_numpy(x, h0, Wx, Wh, b)

dx_num = eval_numerical_gradient_array(fx, x, dh)
dh0_num = eval_numerical_gradient_array(fh0, h0, dh)
dWx_num = eval_numerical_gradient_array(fWx, Wx, dh)
dWh_num = eval_numerical_gradient_array(fWh, Wh, dh)
db_num = eval_numerical_gradient_array(fb, b, dh)

# 你应该看到数量级为1e-6或更小的误差
print('dx error: ', rel_error(dx_num, dx))
print('dh0 error: ', rel_error(dh0_num, dh0))
print('dWx error: ', rel_error(dWx_num, dWx))
print('dWh error: ', rel_error(dWh_num, dWh))
print('db error: ', rel_error(db_num, db))

# 词嵌入：前向传播
在深度学习系统中，我们通常使用向量来表示单词。词汇表中的每个单词将与一个向量相关联，这些向量将与系统的其余部分一起学习。

在文件`cs231n/rnn_layers_pytorch.py`中，实现函数`word_embedding_forward`，将单词（用整数表示）转换为向量。运行以下代码检查你的实现。你应该看到数量级为`e-8`或更小的误差。

In [ ]:
N, T, V, D = 2, 4, 5, 3

x = torch.from_numpy(np.asarray([[0, 3, 1, 2], [2, 1, 0, 3]]))
W = torch.from_numpy(np.linspace(0, 1, num=V*D).reshape(V, D))

out = word_embedding_forward(x, W).numpy()
expected_out = np.asarray([
 [[ 0.,          0.07142857,  0.14285714],
  [ 0.64285714,  0.71428571,  0.78571429],
  [ 0.21428571,  0.28571429,  0.35714286],
  [ 0.42857143,  0.5,         0.57142857]],
 [[ 0.42857143,  0.5,         0.57142857],
  [ 0.21428571,  0.28571429,  0.35714286],
  [ 0.,          0.07142857,  0.14285714],
  [ 0.64285714,  0.71428571,  0.78571429]]])

print('out error: ', rel_error(expected_out, out))

# 词嵌入：反向传播
与之前一样，我们可以使用数值梯度检查器验证pytorch autograd的反向传播。如果你想的话，也可以尝试自己实现`word_embedding_backward`。不过本次作业不要求这样做。

In [ ]:
np.random.seed(231)

N, T, V, D = 50, 3, 5, 6
x = torch.from_numpy(np.random.randint(V, size=(N, T)))
W = torch.from_numpy(np.random.randn(V, D))
W.requires_grad_()

out = word_embedding_forward(x, W)
dout = torch.from_numpy(np.random.randn(*out.shape))
out.backward(dout)

dW = W.grad.detach().numpy()
x = x.detach().numpy()
W = W.detach().numpy()
dout = dout.detach().numpy()

def word_embedding_forward_numpy(x, W):
  return word_embedding_forward(
      torch.from_numpy(x),
      torch.from_numpy(W),
  ).numpy()

f = lambda W: word_embedding_forward_numpy(x, W)
dW_num = eval_numerical_gradient_array(f, W, dout)

# You should see an error on the order of 1e-11 or less
print('dW error: ', rel_error(dW, dW_num))

# 时序仿射层
在每个时间步，我们使用仿射函数将RNN在该时间步的隐藏向量转换为词汇表中每个单词的分数。因为这与你第一次作业中实现的仿射层非常相似，我们已经在`temporal_affine_forward`中为你提供了这个函数。运行以下代码对实现进行数值梯度检查。你应该看到数量级为`e-9`或更小的误差。

In [ ]:
np.random.seed(231)

# Gradient check for temporal affine layer
N, T, D, M = 2, 3, 4, 5
x = torch.from_numpy(np.random.randn(N, T, D))
w = torch.from_numpy(np.random.randn(D, M))
b = torch.from_numpy(np.random.randn(M))

for tensor in [x, w, b]:
  tensor.requires_grad_()
out = temporal_affine_forward(x, w, b)
dout = torch.from_numpy(np.random.randn(*out.shape))
out.backward(dout)

dx = x.grad.detach().numpy()
dw = w.grad.detach().numpy()
db = b.grad.detach().numpy()

x = x.detach().numpy()
w = w.detach().numpy()
b = b.detach().numpy()
dout = dout.detach().numpy()

def temporal_affine_forward_numpy(x, w, b):
  return temporal_affine_forward(
      torch.from_numpy(x),
      torch.from_numpy(w),
      torch.from_numpy(b),
  ).numpy()

fx = lambda x: temporal_affine_forward_numpy(x, w, b)
fw = lambda w: temporal_affine_forward_numpy(x, w, b)
fb = lambda b: temporal_affine_forward_numpy(x, w, b)

dx_num = eval_numerical_gradient_array(fx, x, dout)
dw_num = eval_numerical_gradient_array(fw, w, dout)
db_num = eval_numerical_gradient_array(fb, b, dout)

print('dx error: ', rel_error(dx_num, dx))
print('dw error: ', rel_error(dw_num, dw))
print('db error: ', rel_error(db_num, db))

# 时序Softmax损失
在RNN语言模型中，我们在每个时间步为词汇表中的每个单词生成一个分数。我们知道每个时间步的真实单词，因此使用softmax损失函数来计算每个时间步的损失和梯度。我们将时间上的损失求和，并在小批量上取平均。

然而有一个细节需要注意：由于我们处理的是小批量数据，不同的描述可能具有不同的长度，我们在每个描述的末尾添加`<NULL>`标记，使它们都具有相同的长度。我们不希望这些`<NULL>`标记计入损失或梯度，因此除了分数和真实标签外，我们的损失函数还接受一个`mask`数组，告诉它哪些分数元素计入损失。

由于这与你第一次作业中实现的softmax损失函数非常相似，我们已经为你实现了这个损失函数；查看文件`cs231n/rnn_layers_pytorch.py`中的`temporal_softmax_loss`函数。

运行以下单元格对损失进行合理性检查并对函数进行数值梯度检查。你应该看到dx的误差数量级为`e-7`或更小。

In [ ]:
# 对时序softmax损失进行合理性检查
from cs231n.rnn_layers_pytorch import temporal_softmax_loss

N, T, V = 100, 1, 10

def check_loss(N, T, V, p):
    x = 0.001 * torch.from_numpy(np.random.randn(N, T, V))
    y = torch.from_numpy(np.random.randint(V, size=(N, T)))
    mask = torch.from_numpy(np.random.rand(N, T)) <= p
    print(temporal_softmax_loss(x, y, mask).item())

check_loss(100, 1, 10, 1.0)   # 应该约为2.3
check_loss(100, 10, 10, 1.0)  # 应该约为23
check_loss(5000, 10, 10, 0.1) # 应该在2.2-2.4之间

# 对时序softmax损失进行梯度检查
np.random.seed(231231)
N, T, V = 7, 8, 9

x = torch.from_numpy(np.random.randn(N, T, V))
y = torch.from_numpy(np.random.randint(V, size=(N, T)))
mask = torch.from_numpy(np.random.rand(N, T) > 0.5)

x.requires_grad_()
loss = temporal_softmax_loss(x, y, mask, verbose=False)
loss.backward()
dx = x.grad.detach().numpy()
x = x.detach().numpy()
dx_num = eval_numerical_gradient(
    lambda x: temporal_softmax_loss(torch.from_numpy(x), y, mask), x, verbose=False)

print('dx error: ', rel_error(dx, dx_num))

# 用于图像描述的RNN
现在你已经实现了必要的层，可以将它们组合起来构建一个图像描述模型。打开文件`cs231n/classifiers/rnn_pytorch.py`并查看`CaptioningRNN`类。

在`loss`函数中实现模型的前向传播。目前你只需要实现`cell_type='rnn'`的情况（基础RNN）；你稍后将实现LSTM的情况。完成后，运行以下代码使用一个小测试用例检查你的前向传播；你应该看到数量级为`e-10`或更小的误差。

In [ ]:
N, D, W, H = 10, 20, 30, 40
word_to_idx = {'<NULL>': 0, 'cat': 2, 'dog': 3}
V = len(word_to_idx)
T = 13

model = CaptioningRNN(
    word_to_idx,
    input_dim=D,
    wordvec_dim=W,
    hidden_dim=H,
    cell_type='rnn',
    dtype=torch.float64
)

# Set all model parameters to fixed values
for k, v in model.params.items():
    model.params[k] = torch.from_numpy(
        np.linspace(-1.4, 1.3, num=v.numel()).reshape(*v.shape))

features = torch.from_numpy(np.linspace(-1.5, 0.3, num=(N * D)).reshape(N, D))
captions = torch.from_numpy((np.arange(N * T) % V).reshape(N, T))

loss = model.loss(features, captions).item()
expected_loss = 9.83235591003

print('loss: ', loss)
print('expected loss: ', expected_loss)
print('difference: ', abs(loss - expected_loss))

运行以下单元格对`CaptioningRNN`类进行数值梯度检查；你应该看到数量级为`e-6`或更小的误差。

In [ ]:
np.random.seed(231)
torch.manual_seed(231)

batch_size = 2
timesteps = 3
input_dim = 4
wordvec_dim = 5
hidden_dim = 6
word_to_idx = {'<NULL>': 0, 'cat': 2, 'dog': 3}
vocab_size = len(word_to_idx)

captions = torch.from_numpy(np.random.randint(vocab_size, size=(batch_size, timesteps)))
features = torch.from_numpy(np.random.randn(batch_size, input_dim))

model = CaptioningRNN(
    word_to_idx,
    input_dim=input_dim,
    wordvec_dim=wordvec_dim,
    hidden_dim=hidden_dim,
    cell_type='rnn',
    dtype=torch.float64,
)

for k, v in model.params.items():
  v.requires_grad_()
loss = model.loss(features, captions)
loss.backward()
grads = {k: v.grad.detach().numpy() for k, v in model.params.items()}
for k, v in model.params.items():
  v.requires_grad_(False)

for param_name in sorted(grads.keys()):
    def fn(val):
      model.params[param_name] = torch.from_numpy(val)
      ret = model.loss(features, captions).numpy()
      return ret

    param_grad_num = eval_numerical_gradient(
        fn, model.params[param_name].numpy(), verbose=False, h=1e-6)

    e = rel_error(param_grad_num, grads[param_name])
    print('%s relative error: %e' % (param_name, e))

# 在小数据上过拟合RNN描述模型
类似于我们在上一次作业中用于训练图像分类模型的`Solver`类，在本次作业中我们使用`CaptioningSolverPytorch`类来训练图像描述模型。打开文件`cs231n/captioning_solver_pytorch.py`并通读`CaptioningSolverPytorch`类；它应该看起来非常熟悉。

熟悉API后，运行以下代码确保你的模型在100个训练样本的小样本上过拟合。你应该看到最终损失小于0.1。

In [ ]:
np.random.seed(231)
torch.manual_seed(231)

small_data = load_coco_data(max_train=50)

small_rnn_model = CaptioningRNN(
    cell_type='rnn',
    word_to_idx=data['word_to_idx'],
    input_dim=data['train_features'].shape[1],
    hidden_dim=512,
    wordvec_dim=256,
)

small_rnn_solver = CaptioningSolverPytorch(
    small_rnn_model, small_data,
    num_epochs=50,
    batch_size=25,
    learning_rate=5e-3,
    verbose=True, print_every=10,
)

small_rnn_solver.train()

# Plot the training losses.
plt.plot(small_rnn_solver.loss_history)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training loss history')
plt.show()

打印最终训练损失。你应该看到最终损失小于0.1。

In [ ]:
print('Final loss: ', small_rnn_solver.loss_history[-1])

# 测试时的RNN采样
与分类模型不同，图像描述模型在训练时和测试时的行为非常不同。在训练时，我们可以访问真实描述，因此我们在每个时间步将真实单词作为输入馈送给RNN。在测试时，我们在每个时间步从词汇表上的分布中采样，并将采样结果作为下一个时间步的输入馈送给RNN。

在文件`cs231n/classifiers/rnn_pytorch.py`中，实现测试时采样的`sample`方法。完成后，运行以下代码从你的过拟合模型中在训练数据和验证数据上进行采样。训练数据上的采样结果应该非常好。然而，验证数据上的采样结果可能没有意义。

In [ ]:
# 如果出现错误，说明URL已不存在，不用担心！
# 你可以重新采样任意多次。
for split in ['train', 'val']:
    minibatch = sample_coco_minibatch(small_data, split=split, batch_size=2)
    gt_captions, features, urls = minibatch
    gt_captions = decode_captions(gt_captions, data['idx_to_word'])

    sample_captions = small_rnn_model.sample(torch.from_numpy(features)).numpy()
    sample_captions = decode_captions(sample_captions, data['idx_to_word'])

    for gt_caption, sample_caption, url in zip(gt_captions, sample_captions, urls):
        img = image_from_url(url)
        # 跳过缺失的URL。
        if img is None: continue
        plt.imshow(img)
        plt.title('%s\n%s\nGT:%s' % (split, sample_caption, gt_caption))
        plt.axis('off')
        plt.show()

# 内联问题1

在我们当前的图像描述设置中，RNN语言模型在每个时间步生成一个单词作为输出。然而，另一种提出该问题的方式是训练网络在_字符_（例如'a'、'b'等）上操作，而不是在单词上操作，这样在每个时间步，它接收前一个字符作为输入并尝试预测序列中的下一个字符。例如，网络可能会生成如下描述：

'A', ' ', 'c', 'a', 't', ' ', 'o', 'n', ' ', 'a', ' ', 'b', 'e', 'd'

你能描述一个使用字符级RNN的图像描述模型的优势吗？你也能描述一个劣势吗？提示：有几种有效的答案，但比较词级和字符级模型的参数空间可能是有用的。

**你的答案：**
